# Workshop 3. Hypothesis tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture03/workshop03.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the promise test from the lecture in code, fully worked: run it, answer the questions and change one thing.
Tasks 2 and 3 apply the same steps to new questions on the RetailCorp data, starting from an empty cell. Task 4 uses
a new dataset, an airline passenger survey. Then do the short problems, which use small numbers and no data, much like
a question at the oral exam. The extension tasks are for those who have time left.

Under each core task there are three hints: the first names the statistical step, the second names the functions, and the
third gives the structure of the code as comments. Try the task before you open them. The code patterns below, the
slides and the lecture notebook are your main references. You are welcome to use an LLM, but every task can be
solved without one, and you should be able to explain the statistics in your answers yourself.

In every task, write H<sub>0</sub> and H<sub>1</sub> before you compute anything, and decide from the question whether
the test is one-sided or two-sided.

## Toolbox

The functions you need today. The lecture notebook shows all of them in use.

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std(ddof=1)`, `.var(ddof=1)` | Mean, standard deviation and variance of a column, dividing by n − 1 |
| `len(x)`, `x.sum()` | Number of observations, number of 1s in a 0/1 column |
| `df.loc[df["col"] == "value", "other"]` | The column `other` for the rows where `col` has a given value |
| `np.sqrt(x)` | Square root |
| `stats.t.ppf(1 - alpha, df=n - 1)` | Critical value with α above it in the t distribution; `stats.t.ppf(alpha, df=n - 1)` has α below it. Likewise `stats.chi2.ppf(q, df=n - 1)`, and `stats.norm.ppf(q)` without `df` |
| `stats.t.sf(t, df=n - 1)` | Area above t: the p-value when H<sub>1</sub> has >. Likewise `stats.chi2.sf(chi2, df=n - 1)` and `stats.norm.sf(z)` |
| `stats.t.cdf(t, df=n - 1)` | Area below t: the p-value when H<sub>1</sub> has <. Likewise `stats.chi2.cdf` and `stats.norm.cdf(z)` |
| `stats.ttest_1samp(x, popmean=mu0, alternative="less")` | The whole t test from the data, as a check |
| `stats.skew(x)`, `plt.hist(x, bins=20)` | Skewness and histogram, to check normality |
| `pd.read_csv(url)` | Load a CSV file from a web address |

## Code patterns

Three complete patterns for today, one for each test. Copy a pattern into a task cell, replace `df`, `"column"` and
the value in H<sub>0</sub>, and choose the p-value line that matches your H<sub>1</sub>. The patterns are templates;
they do not run as they stand.

**Test for a mean (t)**

```python
x = df["column"]
n = len(x)
mu0 = 100                                   # the value in H0
alpha = 0.05
t_stat = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(n))
# H1: mu > mu0   critical value t_crit, p-value: area above t_stat
t_crit = stats.t.ppf(1 - alpha, df=n - 1)
p = stats.t.sf(t_stat, df=n - 1)
# H1: mu < mu0   critical value with alpha below it (negative), p-value: area below t_stat
# t_crit = stats.t.ppf(alpha, df=n - 1)
# p = stats.t.cdf(t_stat, df=n - 1)
# H1: mu != mu0  critical values -/+ stats.t.ppf(1 - alpha / 2, df=n - 1), p-value: both tails
# p = 2 * stats.t.sf(abs(t_stat), df=n - 1)
print(f"t = {t_stat:.3f}, crit = {t_crit:.3f}, p = {p:.4f}")
```

**Test for a proportion (z, standard error with P0 as in Newbold)**

```python
x = df["column"]                            # a 0/1 column, or a condition such as df["column"] > value
n = len(x)
p0 = 0.10                                   # the value in H0
p_hat = x.mean()
print(n * p0 * (1 - p0))                    # the condition: should be above 5
z = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n)
p = stats.norm.sf(z)                        # H1: P > p0; use stats.norm.cdf(z) for H1: P < p0
print(f"p_hat = {p_hat:.3f}, z = {z:.3f}, p = {p:.4f}")
```

**Test for a variance (chi-square, needs a normal population)**

```python
x = df["column"]
n = len(x)
sigma0 = 5                                  # the standard deviation in H0
chi2 = (n - 1) * x.var(ddof=1) / sigma0**2
crit = stats.chi2.ppf(1 - alpha, df=n - 1)  # H1: sigma > sigma0, reject if chi2 > crit
p = stats.chi2.sf(chi2, df=n - 1)
# H1: sigma < sigma0   reject if chi2 < stats.chi2.ppf(alpha, df=n - 1), p = stats.chi2.cdf(chi2, df=n - 1)
print(f"chi2 = {chi2:.2f}, crit = {crit:.2f}, p = {p:.4f}")
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
alpha = 0.05

## Core tasks

### Task 1. The promise test in code (worked)

RetailCorp promises that at most 5% of orders arrive later than 48 hours. Does the study of 200 orders give strong
evidence that it breaks the promise? The hypotheses are H<sub>0</sub>: P ≤ 0.05 and H<sub>1</sub>: P > 0.05. The cell
below runs the test exactly as in the lecture. Run it and answer the questions.

1. Which number in the output is the standard error? Why is it computed with 0.05 and not with p̂ = 0.080?
2. Change `orders = sample` to `orders = sample[sample["warehouse"] == "Copenhagen"]` and run the cell again. Look at
the condition first. Should RetailCorp use this test for Copenhagen?
3. Write the result for all 200 orders in one or two sentences for a manager at RetailCorp.

In [ ]:
orders = sample                             # change in question 2
p0 = 0.05                                   # the share in H0
n = len(orders)
p_hat = orders["late"].mean()
print(f"n = {n}, late = {orders['late'].sum()}, p_hat = {p_hat:.3f}")
print(f"Condition n * p0 * (1 - p0): {n * p0 * (1 - p0):.1f}")

se = np.sqrt(p0 * (1 - p0) / n)             # standard error with P0, the value in H0
z = (p_hat - p0) / se
z_crit = stats.norm.ppf(1 - alpha)
p = stats.norm.sf(z)                        # area above z, since H1 has >
print(f"Standard error = {se:.5f}")
print(f"z = {z:.3f}, crit = {z_crit:.3f}, p = {p:.3f}")

**Your answer:**

### Task 2. Is the average basket value below the budget?

RetailCorp's budget for next year assumes an average basket value of 650 DKK. The finance team worries that the
average is lower. Test this at α = 0.05 with the 200 orders, using `order_value_dkk`.

**Answer in words.** State H<sub>0</sub> and H<sub>1</sub>, the decision, and the result for the finance team.
Basket values are strongly skewed (Lecture 1). Can the t test still be used here? Explain why.

<details>
<summary>Hint 1</summary>

This is the t test for a mean with σ unknown, one-sided. The worry is that the average is *lower*, so
H<sub>0</sub>: μ ≥ 650 and H<sub>1</sub>: μ < 650. The rejection region is in the lower tail.
</details>

<details>
<summary>Hint 2</summary>

`.mean()`, `.std(ddof=1)`, `len`, `np.sqrt`, `stats.t.ppf(alpha, df=n - 1)` for the lower critical value (or
`-stats.t.ppf(1 - alpha, df=n - 1)`, the same number), and `stats.t.cdf` for the p-value. The first code pattern shows
the lines for H<sub>1</sub> with <.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the column order_value_dkk
# n, the mean and the standard deviation (ddof=1)
# t = (mean - 650) / (s / sqrt(n))
# the lower critical value: reject H0 if t is below it
# the p-value: the area below t
# print t, the critical value and p
```
</details>

**Your answer:**

### Task 3. Are deliveries from Aarhus too spread out?

The operations target says that the standard deviation of delivery times should be at most 7.3 hours (in the lecture
we tested all orders and Copenhagen). Test the target for orders shipped from Aarhus at α = 0.05. The test assumes
a normal population, so look at a histogram of the Aarhus delivery times first.

**Answer in words.** State H<sub>0</sub> and H<sub>1</sub>. Is the normality assumption reasonable? What does the
result tell the Aarhus warehouse, and what does it not tell?

<details>
<summary>Hint 1</summary>

This is the chi-square test for a variance with H<sub>0</sub>: σ² ≤ 7.3² and H<sub>1</sub>: σ² > 7.3², one-sided
in the upper tail. Select the Aarhus orders first.
</details>

<details>
<summary>Hint 2</summary>

`sample.loc[sample["warehouse"] == "Aarhus", "delivery_hours"]`, `plt.hist`, `stats.skew`, `.var(ddof=1)`,
`stats.chi2.ppf(1 - alpha, df=n - 1)` and `stats.chi2.sf`. See the third code pattern.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the delivery times of the Aarhus orders
# histogram and skewness to check the normality assumption
# n and the sample variance (ddof=1)
# chi2 = (n - 1) * s^2 / 7.3^2
# the critical value with alpha above it, and the p-value (area above chi2)
# print chi2, the critical value and p
```
</details>

**Your answer:**

### Task 4. A new dataset: an airline passenger survey

So far all data came from RetailCorp. This task uses a public survey of about 26,000 airline passengers from Kaggle
(Airline Passenger Satisfaction). Passengers rated services such as seat comfort, baggage handling and check-in on a
scale from 1 to 5, and said whether they were satisfied overall. In some rating columns 0 means "not rated"; today we
use columns without zeros. The data README in the course repository describes the file.

**Loading the file (optional, an offer to learn).** In your own projects the data will not come with a ready-made
web address. If you want to practice, find the address of the file yourself:

1. Open [github.com/aaubs/AQM-2026](https://github.com/aaubs/AQM-2026) in your browser and click the folder `data`.
2. Click the file `airline_survey.csv`.
3. GitHub shows the file page. Click the button **Raw** at the top right of the file (for a large file GitHub may
show a link **View raw** in the middle of the page instead).
4. The browser now shows the plain text of the file. Copy the address from the address bar. It starts with
`https://raw.githubusercontent.com/`.
5. Paste the address between the quotation marks in the cell below and run it.

Only the raw address works with `pd.read_csv`. The normal GitHub address (starting with `https://github.com/`) points
to a web page about the file, not to the file itself.

If you prefer to skip this step, open the hidden cell below and copy the address from there.

<details>
<summary>The address of the file</summary>

```python
airline_url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/airline_survey.csv"
```
</details>

In [ ]:
airline_url = ""        # paste the raw address of airline_survey.csv here
airline = pd.read_csv(airline_url)
print(airline.shape)
airline.head()

The cell gives an error until you have pasted an address.

**A quick check of the file.** Run the cell below before you test anything: how many passengers and columns are
there, and do the ratings lie between 1 and 5?

In [ ]:
columns = ["Seat comfort", "Baggage handling", "Checkin service", "Gate location"]
airline[columns].describe().round(2)

**The task.** The airline's service target is an average seat comfort rating of at least 3.5. Does the survey give
strong evidence that the average seat comfort rating is below the target? Test at α = 0.05, using the column
`Seat comfort`.

**Answer in words.** State H<sub>0</sub> and H<sub>1</sub>, the decision, and the result for the airline's service
manager. How far below 3.5 is the sample average, on a scale from 1 to 5?

<details>
<summary>Hint 1</summary>

This is the t test for a mean, one-sided in the lower tail: H<sub>0</sub>: μ ≥ 3.5 and H<sub>1</sub>: μ < 3.5. The
steps are the same as in Task 2, on a different dataset.
</details>

<details>
<summary>Hint 2</summary>

`airline["Seat comfort"]`, `.mean()`, `.std(ddof=1)`, `len`, `np.sqrt`, `stats.t.ppf(alpha, df=n - 1)`,
`stats.t.cdf`. Check with `stats.ttest_1samp(x, popmean=3.5, alternative="less")`.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the column Seat comfort
# n, the mean and the standard deviation (ddof=1)
# t = (mean - 3.5) / (s / sqrt(n))
# the lower critical value and the p-value (area below t)
# print n, the mean, t, the critical value and p
```
</details>

**Your answer:**

## Short problems

These problems use small numbers and no data. Solve them with the formulas from the slides, and use Python only to
look up critical values and p-values, or a table if you prefer. Write your reasoning in words, as you would explain it
at the exam.

### Problem 1. A bakery's loaves

A bakery says its loaves weigh 800 g on average. A food inspector weighs 16 randomly chosen loaves: the mean is
792 g and the standard deviation 14 g. Loaf weights are roughly normal. The inspector wants to know whether the
average differs from 800 g, in either direction.

1. State H<sub>0</sub> and H<sub>1</sub>, and compute the test statistic.
2. Find the critical value at α = 0.05 and make the decision. What is the p-value?
3. Why is the test two-sided here?

<details>
<summary>Hint 1</summary>

t = (x̄ − μ<sub>0</sub>) / (s/√n) with 15 degrees of freedom. Two-sided: compare |t| with t<sub>15, 0.025</sub>, and
double the tail area for the p-value.
</details>

<details>
<summary>Hint 2</summary>

`stats.t.ppf(0.975, df=15)` and `2 * stats.t.sf(abs(t), df=15)`.
</details>

**Your answer:**

### Problem 2. AI use in small firms

A ministry report states that 25% of small and medium-sized firms use AI tools. A researcher surveys a random sample
of 400 firms in one region, and 88 of them use AI tools. She wants to know whether the share in her region is lower
than 25%.

1. State H<sub>0</sub> and H<sub>1</sub>, and check the condition.
2. Compute z and the p-value. What is the decision at α = 0.05? At α = 0.10?
3. Explain which standard error you used and why.

<details>
<summary>Hint 1</summary>

The test for a proportion, one-sided in the lower tail: H<sub>0</sub>: P ≥ 0.25, H<sub>1</sub>: P < 0.25. The standard
error uses P<sub>0</sub> = 0.25.
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt(0.25 * 0.75 / 400)`, then `stats.norm.cdf(z)` for the p-value.
</details>

**Your answer:**

### Problem 3. A more precise filling machine

A new machine fills bags of coffee. The old machine had a standard deviation of 2 g, and the new one should do better.
In a random sample of 10 bags from the new machine the standard deviation is 1.2 g. The weights are normally
distributed.

1. State H<sub>0</sub> and H<sub>1</sub> for "the new machine is more precise than the old one".
2. Compute the test statistic and make the decision at α = 0.05. What is the p-value?
3. Why does the problem need to say that the weights are normal?

<details>
<summary>Hint 1</summary>

H<sub>0</sub>: σ² ≥ 4 and H<sub>1</sub>: σ² < 4. The rejection region is in the lower tail: reject H<sub>0</sub> if
χ² is below χ²<sub>9, 0.95</sub>, the value with 0.95 above it.
</details>

<details>
<summary>Hint 2</summary>

`stats.chi2.ppf(0.05, df=9)` gives the value with 0.05 below it, which is χ²<sub>9, 0.95</sub> in Newbold's notation.
The p-value is `stats.chi2.cdf(chi2, df=9)`.
</details>

**Your answer:**

### Problem 4. Reading a p-value

A test gives p = 0.03.

1. What is the decision at α = 0.10, 0.05 and 0.01?
2. What is the smallest significance level at which H<sub>0</sub> would be rejected?
3. A colleague says: "So there is a 3% chance that H<sub>0</sub> is true." Explain what is wrong, and say what
p = 0.03 does mean.

**Your answer:**

## Extension tasks

### Extension 1. Are half of the passengers satisfied?

In the airline data, the column `satisfaction` is either "satisfied" or "neutral or dissatisfied". Test whether the
share of satisfied passengers differs from 50%, at α = 0.05.

**Answer in words.** State H<sub>0</sub> and H<sub>1</sub>, check the condition, and give the result. Note what
"satisfied" is compared with.

<details>
<summary>Hint 1</summary>

A two-sided test for a proportion: H<sub>0</sub>: P = 0.5, H<sub>1</sub>: P ≠ 0.5. The share is the mean of the
condition `airline["satisfaction"] == "satisfied"`.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern; for the two-sided p-value use `2 * stats.norm.sf(abs(z))`.
</details>

**Your answer:**

### Extension 2. Why does every test reject with 26,000 passengers?

In Task 4 the average seat comfort was only 0.05 points below 3.5, yet the p-value was tiny. Draw a random sample of
200 passengers from the airline data (`airline.sample(n=200, random_state=1)`) and repeat the Task 4 test on it.

**Answer in words.** Compare the two results. What does the size of the sample do to the standard error, and what is
the difference between a statistically significant result and an important one?

<details>
<summary>Hint 1</summary>

Repeat the Task 4 code on the smaller sample and compare the standard errors s/√n.
</details>

<details>
<summary>Hint 2</summary>

`small = airline.sample(n=200, random_state=1)`, then the same lines as in Task 4 on `small["Seat comfort"]`.
</details>

**Your answer:**

### Extension 3. The promise test with statsmodels

The statsmodels package has a function for the test of a proportion:

```python
from statsmodels.stats.proportion import proportions_ztest
proportions_ztest(count, nobs, value=0.05, alternative="larger")
```

Run it for the promise test (16 late orders out of 200), first as written, then with `prop_var=0.05` added.

**Answer in words.** Compare the two results with Task 1. Why do they differ, and which one matches the lecture?
Explain which standard error each one uses.

<details>
<summary>Hint 1</summary>

One of the two versions computes the standard error with p̂, the other with P<sub>0</sub> = 0.05.
</details>

<details>
<summary>Hint 2</summary>

`count = 16`, `nobs = 200`. The function returns z and the p-value.
</details>

**Your answer:**